# NetCDF Temperature Extraction to CSV (Bremen)

## Main objective

Extract local daily mean temperature from a large E-OBS NetCDF file, convert the values to degC when required, save the local result as CSV, and perform basic pandas quality checks.


In [ ]:
# Import the main libraries used in this notebook
# pathlib handles file paths safely on Windows
# numpy supports numerical checks
# pandas is used for tabular checks and CSV export
# xarray reads NetCDF data

from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr

In [ ]:
# Define project paths
# This uses the same project path as the precipitation notebook

project_root = Path(r"C:\Users\Amanda\Desktop\ISU\6. Semester\Virtual Reality and Optimization\vro_project")
nc_path = project_root / "data" / "raw" / "tg_ens_mean_0.1deg_reg_v33.0e.nc"
output_csv = project_root / "data" / "processed" / "bremen_daily_mean_temperature.csv"

# Select study location: Bremen, Germany
# These coordinates are used to find the nearest E-OBS grid point

target_lat = 53.071895
target_lon = 8.794528

# Define a small regional box around Bremen: Neustadtswall

bbox = {
    "min_lat": target_lat - 0.2,
    "max_lat": target_lat + 0.2,
    "min_lon": target_lon - 0.25,
    "max_lon": target_lon + 0.25,
}

print("Project folder:", project_root)
print("NetCDF file:", nc_path)
print("NetCDF file exists:", nc_path.exists())
print("Output CSV path:", output_csv)

if not nc_path.exists():
    raise FileNotFoundError(
        f"NetCDF file not found: {nc_path}\\n"
        "Check the project folder and NetCDF filename."
    )

Project folder: C:\Users\Amanda\Desktop\ISU\6. Semester\Virtual Reality and Optimization\Project
NetCDF file: C:\Users\Amanda\Desktop\ISU\6. Semester\Virtual Reality and Optimization\Project\data\raw\tg_ens_mean_0.1deg_reg_v33.0e.nc
NetCDF file exists: True
Output CSV path: C:\Users\Amanda\Desktop\ISU\6. Semester\Virtual Reality and Optimization\Project\data\processed\bremen_daily_mean_temperature.csv


In [ ]:
# Open the NetCDF dataset
# This does not convert the complete dataset to pandas
# First inspect the structure so that no incorrect assumptions are made

ds = xr.open_dataset(nc_path)

print("Data variables:", list(ds.data_vars))
print("Dimensions:", dict(ds.sizes))
print("Coordinates:", list(ds.coords))
print("Global attribute keys:", list(ds.attrs.keys()))

Data variables: ['tg']
Dimensions: {'time': 27759, 'latitude': 465, 'longitude': 705}
Coordinates: ['longitude', 'latitude', 'time']
Global attribute keys: ['E-OBS_version', 'Conventions', 'References', 'history', 'NCO']


In [ ]:
# Detect the temperature variable safely
# If there is only one data variable, use it
# If there are several variables, search variable names and metadata

if len(ds.data_vars) == 1:
    temperature_var = list(ds.data_vars)[0]
else:
    candidates = []
    for name in ds.data_vars:
        name_text = name.lower()
        attrs_text = " ".join(
            str(value).lower() for value in ds[name].attrs.values()
        )
        if (
            "temperature" in name_text
            or "temperature" in attrs_text
            or "mean temperature" in attrs_text
        ):
            candidates.append(name)

    if not candidates:
        raise ValueError(
            "No clear temperature variable was detected. "
            "Inspect the available data variables manually."
        )

    temperature_var = candidates[0]

print("Selected temperature variable:", temperature_var)
print("Variable attributes:")
for key, value in ds[temperature_var].attrs.items():
    print(f"  {key}: {value}")

Selected temperature variable: tg
Variable attributes:
  units: Celsius
  long_name: mean temperature
  standard_name: air_temperature


In [ ]:
# Detect coordinate names for time, latitude, and longitude

time_name = next(
    coordinate for coordinate in ds.coords
    if "time" in coordinate.lower()
)

lat_name = next(
    coordinate for coordinate in ds.coords
    if coordinate.lower() in ("lat", "latitude", "y")
)

lon_name = next(
    coordinate for coordinate in ds.coords
    if coordinate.lower() in ("lon", "longitude", "x")
)

print("Detected coordinate names:")
print("  time ->", time_name)
print("  latitude ->", lat_name)
print("  longitude ->", lon_name)

Detected coordinate names:
  time -> time
  latitude -> latitude
  longitude -> longitude


In [ ]:
# Confirm the actual time coverage

start_date = pd.to_datetime(ds[time_name].values[0])
end_date = pd.to_datetime(ds[time_name].values[-1])
number_of_days = int(ds.sizes[time_name])
approximate_years = (end_date - start_date).days / 365.25

print("First date:", start_date.date())
print("Last date:", end_date.date())
print("Number of daily time steps:", number_of_days)
print(f"Approximate period length: {approximate_years:.1f} years")

if approximate_years < 30:
    print(
        "Warning: The file contains fewer than 30 years of data. "
        "It can be used for descriptive analysis, but it is too short "
        "for a strong long-term climate trend conclusion."
    )

First date: 1950-01-01
Last date: 2025-12-31
Number of daily time steps: 27759
Approximate period length: 76.0 years


## Compact inspection

The following cells inspect only the structure and small local subsets. The complete European dataset is not converted to pandas.


In [ ]:
# Print a compact xarray summary

print(ds)
print()
print(ds[temperature_var])


<xarray.Dataset> Size: 36GB
Dimensions:    (time: 27759, latitude: 465, longitude: 705)
Coordinates:
  * time       (time) datetime64[ns] 222kB 1950-01-01 1950-01-02 ... 2025-12-31
  * latitude   (latitude) float64 4kB 25.05 25.15 25.25 ... 71.25 71.35 71.45
  * longitude  (longitude) float64 6kB -24.95 -24.85 -24.75 ... 45.35 45.45
Data variables:
    tg         (time, latitude, longitude) float32 36GB ...
Attributes:
    E-OBS_version:  33.0e
    Conventions:    CF-1.4
    References:     http://surfobs.climate.copernicus.eu/dataaccess/access_eo...
    history:        Tue Mar 10 15:21:27 2026: ncks -O --no-abc -d time,0,2775...
    NCO:            netCDF Operators version 5.3.3 (Homepage = http://nco.sf....

<xarray.DataArray 'tg' (time: 27759, latitude: 465, longitude: 705)> Size: 36GB
[9100094175 values with dtype=float32]
Coordinates:
  * time       (time) datetime64[ns] 222kB 1950-01-01 1950-01-02 ... 2025-12-31
  * latitude   (latitude) float64 4kB 25.05 25.15 25.25 ... 71.25 71

In [ ]:
# Create coordinate slices that work with increasing or decreasing coordinates

def coordinate_slice(coordinate, minimum, maximum):
    first_value = float(coordinate.values[0])
    last_value = float(coordinate.values[-1])

    if first_value < last_value:
        return slice(minimum, maximum)

    return slice(maximum, minimum)

lat_slice = coordinate_slice(
    ds[lat_name],
    bbox["min_lat"],
    bbox["max_lat"],
)

lon_slice = coordinate_slice(
    ds[lon_name],
    bbox["min_lon"],
    bbox["max_lon"],
)

print("Latitude ordering:")
print(float(ds[lat_name].values[0]), "to", float(ds[lat_name].values[-1]))

print("Longitude ordering:")
print(float(ds[lon_name].values[0]), "to", float(ds[lon_name].values[-1]))

Latitude ordering:
25.049860609910866 to 71.44986046288214
Longitude ordering:
-24.950139509200024 to 45.44986020916354


In [ ]:
# Inspect one day only for the selected region
# This avoids converting a full European temperature map to pandas

one_day_region = ds[temperature_var].isel(
    {time_name: 0}
).sel(
    {
        lat_name: lat_slice,
        lon_name: lon_slice,
    }
)

one_day_region_df = one_day_region.to_dataframe(
    name="temperature_original_unit"
).reset_index()

print("One-day regional table shape:", one_day_region_df.shape)
display(one_day_region_df.head(10))

One-day regional table shape: (20, 4)


,latitude,longitude,time,temperature_original_unit
0,52.949861,8.54986,1950-01-01,-1.15
1,52.949861,8.64986,1950-01-01,-1.01
2,52.949861,8.74986,1950-01-01,-0.94
3,52.949861,8.84986,1950-01-01,-0.90
4,52.949861,8.94986,1950-01-01,-1.03
5,53.049861,8.54986,1950-01-01,-1.02
6,53.049861,8.64986,1950-01-01,-0.91
7,53.049861,8.74986,1950-01-01,-0.74
8,53.049861,8.84986,1950-01-01,-1.08
9,53.049861,8.94986,1950-01-01,-0.88


In [ ]:
# Extract two local series:
# 1. nearest grid cell to Bremen,
# 2. all grid cells inside the selected regional box

temperature_data = ds[temperature_var]

nearest = temperature_data.sel(
    {
        lat_name: target_lat,
        lon_name: target_lon,
    },
    method="nearest",
)

region = temperature_data.sel(
    {
        lat_name: lat_slice,
        lon_name: lon_slice,
    }
)

nearest_lat = float(nearest[lat_name].values)
nearest_lon = float(nearest[lon_name].values)

regional_cell_count = (
    int(region.sizes[lat_name])
    * int(region.sizes[lon_name])
)

if regional_cell_count == 0:
    raise ValueError(
        "The regional selection contains no grid cells. "
        "Check the bounding box and coordinate ordering."
    )

print(
    f"Requested location: lat={target_lat:.4f}, "
    f"lon={target_lon:.4f}"
)

print(
    f"Nearest selected grid point: lat={nearest_lat:.4f}, "
    f"lon={nearest_lon:.4f}"
)

print("Latitude grid cells:", region.sizes[lat_name])
print("Longitude grid cells:", region.sizes[lon_name])
print("Total regional grid cells:", regional_cell_count)

Requested location: lat=53.0719, lon=8.7945
Nearest selected grid point: lat=53.0499, lon=8.7499
Latitude grid cells: 4
Longitude grid cells: 5
Total regional grid cells: 20


In [ ]:
# Check the original temperature unit and convert to degC when required

original_units = str(
    temperature_data.attrs.get("units", "")
).strip()

units_lower = original_units.lower()

kelvin_units = {
    "k",
    "kelvin",
}

celsius_units = {
    "degc",
    "degree_celsius",
    "degrees_celsius",
    "degree celsius",
    "degrees celsius",
    "celsius",
    "c",
}

print("Original temperature unit:", original_units)

if units_lower in kelvin_units:
    print("Converting temperature from Kelvin to degC.")
    nearest_c = nearest - 273.15
    region_c = region - 273.15

elif units_lower in celsius_units:
    print("Temperature is already stored in degC.")
    nearest_c = nearest
    region_c = region

else:
    raise ValueError(
        f"Unrecognized temperature unit: {original_units}. "
        "Inspect the variable metadata before continuing."
    )

Original temperature unit: Celsius
Temperature is already stored in degC.


In [ ]:
# Convert the extracted xarray objects to pandas tables

nearest_df = nearest_c.to_dataframe(
    name="nearest_grid_temperature_degC"
).reset_index()

regional_mean = region_c.mean(
    dim=(lat_name, lon_name),
    skipna=True,
)

regional_df = regional_mean.to_dataframe(
    name="regional_mean_temperature_degC"
).reset_index()

df = nearest_df[
    [time_name, "nearest_grid_temperature_degC"]
].merge(
    regional_df[
        [time_name, "regional_mean_temperature_degC"]
    ],
    on=time_name,
    how="outer",
)

df = df.rename(columns={time_name: "date"})
df["date"] = pd.to_datetime(df["date"])

# Use the regional mean as the main series
# Both series are kept so that we can compare them

df["selected_temperature_degC"] = (
    df["regional_mean_temperature_degC"]
)

df = df.sort_values("date").reset_index(drop=True)

output_csv.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output_csv, index=False)

print("Saved CSV:", output_csv)

Saved CSV: C:\Users\Amanda\Desktop\ISU\6. Semester\Virtual Reality and Optimization\Project\data\processed\bremen_daily_mean_temperature.csv


In [ ]:
# Initial pandas checks

print("Shape:", df.shape)

print("\nHead:")
display(df.head())

print("\nMissing values per column:")
print(df.isna().sum())

print("\nDescriptive statistics:")
display(
    df[
        [
            "nearest_grid_temperature_degC",
            "regional_mean_temperature_degC",
            "selected_temperature_degC",
        ]
    ].describe()
)

Shape: (27759, 4)

Head:


,date,nearest_grid_temperature_degC,regional_mean_temperature_degC,selected_temperature_degC
0,1950-01-01,-0.74,-0.9695,-0.9695
1,1950-01-02,3.42,3.4190,3.4190
2,1950-01-03,4.55,4.3130,4.3130
3,1950-01-04,0.44,0.4325,0.4325
4,1950-01-05,2.44,2.1995,2.1995



Missing values per column:
date                              0
nearest_grid_temperature_degC     0
regional_mean_temperature_degC    0
selected_temperature_degC         0
dtype: int64

Descriptive statistics:


,nearest_grid_temperature_degC,regional_mean_temperature_degC,selected_temperature_degC
count,27759.000000,27759.000000,27759.000000
mean,9.482107,9.369290,9.369290
std,6.886025,6.840221,6.840221
min,-15.780000,-15.937001,-15.937001
25%,4.490000,4.403500,4.403500
50%,9.639999,9.518499,9.518499
75%,14.790000,14.663749,14.663749
max,29.879999,29.441998,29.441998


In [ ]:
# Additional quality checks
# Negative temperatures are valid and are not treated as errors

duplicate_dates = int(
    df.duplicated(subset=["date"]).sum()
)

expected_dates = pd.date_range(
    df["date"].min(),
    df["date"].max(),
    freq="D",
)

missing_dates = expected_dates.difference(df["date"])

very_low_count = int(
    (df["selected_temperature_degC"] < -60).sum()
)

very_high_count = int(
    (df["selected_temperature_degC"] > 60).sum()
)

print("Duplicate dates:", duplicate_dates)
print("Missing dates count:", len(missing_dates))
print("Values below -60 degC:", very_low_count)
print("Values above 60 degC:", very_high_count)

if len(missing_dates) > 0:
    print(
        "First missing date examples:",
        list(missing_dates[:10]),
    )

Duplicate dates: 0
Missing dates count: 0
Values below -60 degC: 0
Values above 60 degC: 0


In [ ]:
# Compare the nearest-grid series with the regional-average series

comparison = df[
    [
        "nearest_grid_temperature_degC",
        "regional_mean_temperature_degC",
    ]
].corr()

print("Correlation between extraction methods:")
display(comparison)

df["point_region_difference_degC"] = (
    df["nearest_grid_temperature_degC"]
    - df["regional_mean_temperature_degC"]
)

print("Point minus regional mean difference:")
display(
    df["point_region_difference_degC"].describe()
)

# Save the additional comparison column

df.to_csv(output_csv, index=False)
print("Updated CSV saved:", output_csv)

Correlation between extraction methods:


,nearest_grid_temperature_degC,regional_mean_temperature_degC
nearest_grid_temperature_degC,1.0000,0.9995
regional_mean_temperature_degC,0.9995,1.0000


Point minus regional mean difference:


count    27759.000000
mean         0.112816
std          0.221764
min         -1.196500
25%         -0.023000
50%          0.100000
75%          0.233501
max          1.336501
Name: point_region_difference_degC, dtype: float64

Updated CSV saved: C:\Users\Amanda\Desktop\ISU\6. Semester\Virtual Reality and Optimization\Project\data\processed\bremen_daily_mean_temperature.csv


## Merging of the datasets using the date column

In [ ]:
# Define paths of files

df_rain = pd.read_csv(r'C:\Users\Amanda\Desktop\ISU\6. Semester\Virtual Reality and Optimization\Project\data\processed\bremen_daily_precipitation.csv')
df_temp = pd.read_csv(r'C:\Users\Amanda\Desktop\ISU\6. Semester\Virtual Reality and Optimization\Project\data\processed\bremen_daily_mean_temperature.csv')

# Merge them

df_merged = pd.merge(df_rain, df_temp, on='date', how= 'inner')

# Save merged data

df_merged.to_csv('merged_data.csv', index=False)

In [ ]:
# Close the dataset to release resources

ds.close()
print("Dataset closed.")

Dataset closed.
